<a href="https://colab.research.google.com/github/Agarshan29/Sample/blob/main/Neuro_Detect.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import tensorflow as tf
from tensorflow.keras import layers, models
import os
import matplotlib.pyplot as plt
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras.preprocessing import image
import numpy as np
from sklearn.model_selection import train_test_split

In [3]:
data_path = "/content/drive/MyDrive/Training"

In [4]:
train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=10,
    width_shift_range=0.1,
    height_shift_range=0.1,
    shear_range=0.1,
    zoom_range=0.1,
    horizontal_flip=True,
    validation_split=0.2  # Split for validation
)


In [5]:
train_generator = train_datagen.flow_from_directory(
    data_path,
    target_size=(227, 227),
    batch_size=32,
    class_mode='categorical',
    subset='training'
)

validation_generator = train_datagen.flow_from_directory(
    data_path,
    target_size=(227, 227),
    batch_size=32,
    class_mode='categorical',
    subset='validation'
)


Found 5627 images belonging to 4 classes.
Found 1406 images belonging to 4 classes.


In [6]:
def build_model(num_classes):
    model = models.Sequential()
    model.add(layers.Conv2D(64, (7, 7), strides=(2, 2), padding='same', activation='relu', input_shape=(227, 227, 3)))
    model.add(layers.BatchNormalization())
    model.add(layers.MaxPooling2D((3, 3), strides=(2, 2)))

    model.add(layers.Conv2D(128, (5, 5), padding='same', activation='relu'))
    model.add(layers.BatchNormalization())
    model.add(layers.MaxPooling2D((3, 3), strides=(2, 2)))

    model.add(layers.Conv2D(256, (3, 3), padding='same', activation='relu'))
    model.add(layers.BatchNormalization())

    model.add(layers.Conv2D(256, (3, 3), padding='same', activation='relu'))
    model.add(layers.BatchNormalization())

    model.add(layers.Conv2D(512, (3, 3), padding='same', activation='relu'))
    model.add(layers.BatchNormalization())
    model.add(layers.GlobalAveragePooling2D())

    model.add(layers.Dropout(0.5))
    model.add(layers.Dense(256, activation='relu'))
    model.add(layers.Dropout(0.5))
    model.add(layers.Dense(num_classes, activation='softmax'))

    model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
                  loss='categorical_crossentropy',
                  metrics=['accuracy'])
    return model

In [7]:
num_classes = len(train_generator.class_indices)
model = build_model(num_classes)


/usr/local/lib/python3.11/dist-packages/keras/src/layers/convolutional/base_conv.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [ ]:
callbacks = [
    tf.keras.callbacks.CSVLogger("training_log.csv", append=False),
    tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)
]

history = model.fit(
    train_generator,
    epochs=60,
    validation_data=validation_generator,
    callbacks=callbacks
)

Epoch 1/60
176/176 ━━━━━━━━━━━━━━━━━━━━ 0s 12s/step - accuracy: 0.6728 - loss: 0.8305 

In [ ]:
num_classes = len(train_generator.class_indices)
model = build_model(num_classes)

# Add Callback for Detailed Logs
callbacks = [
    tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True)
]

history = model.fit(
    train_generator,
    epochs=20,
    validation_data=validation_generator,
    callbacks=callbacks
)

Epoch 1/20
 84/144 ━━━━━━━━━━━━━━━━━━━━ 7:02 7s/step - accuracy: 0.5796 - loss: 1.0387

KeyboardInterrupt: 